# 05 · Generation + RAGAS — `scale_15K` (15,000 papers)
*(Run on Kaggle with GPU + GEMINI_API_KEY in Kaggle secrets)*

In [ ]:
# ══════════════════════════════════════════════════════════════
# SCALE EXPERIMENT CONFIG  ← only N_PAPERS and SCALE_LABEL change between tiers
# ══════════════════════════════════════════════════════════════
import os, sys, json, random
import numpy as np
import pandas as pd
import torch

SCALE_LABEL   = "scale_15K"        # identifier used in output filenames
N_PAPERS      = 15000              # ← THE ONLY NUMBER THAT CHANGES PER TIER
RANDOM_SEED   = 42               # NEVER change this
CHUNK_SIZE    = 400              # tokens
CHUNK_OVERLAP = 50               # tokens
RRF_K         = 60               # standard RRF parameter
TOP_K_STAGE1  = 50               # candidates sent to reranker
EVAL_K_VALUES = [1, 3, 5, 10, 20]

# ── Hardware (fixed by FIX 1) ─────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT    = "../../1_data"
RESULTS_DIR  = f"../../4_results/{SCALE_LABEL}"
GT_PATH      = f"{DATA_ROOT}/eval/ground_truth.json"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Scale : {SCALE_LABEL} | N = {N_PAPERS:,} | Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# ── Load Gemini key (FIX 2: correct RAGAS wrapper) ────────────────────────────
import os
from dotenv import load_dotenv
load_dotenv("../../.env")
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_KEY:
    # Kaggle: load from secrets
    try:
        from kaggle_secrets import UserSecretsClient
        GEMINI_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
    except:
        pass
assert GEMINI_KEY, "GEMINI_API_KEY not found. Add to .env (local) or Kaggle secrets."
print("Gemini key loaded ✓")

In [ ]:
import google.generativeai as genai
genai.configure(api_key=GEMINI_KEY)
gen_model = genai.GenerativeModel("gemini-1.5-flash")

# Test
resp = gen_model.generate_content("Say hello in one word.")
print("Gemini test:", resp.text)

In [ ]:
# Generate answers for all 50 queries across 3 systems
import json, pickle, numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

df_chunks = pd.read_parquet(f"{RESULTS_DIR}/chunks.parquet")
texts = df_chunks['text'].tolist()
chunk_ids = df_chunks['chunk_id'].tolist()
id_to_text = dict(zip(chunk_ids, texts))
id_to_cord = dict(zip(chunk_ids, df_chunks['cord_uid'].tolist()))

embeddings = np.load(f"{RESULTS_DIR}/bge_m3_embeddings.npy")
model = SentenceTransformer("BAAI/bge-m3", device=DEVICE)
with open(f"{RESULTS_DIR}/bm25_index.pkl", "rb") as f:
    bm25 = pickle.load(f)

with open("../../1_data/eval/queries.json") as f:
    QUERIES = json.load(f)
print(f"Queries: {len(QUERIES)} | Chunks: {len(texts):,}")

In [ ]:
def generate_answer(query, context_chunks):
    context = "\n\n".join([f"[{i+1}] {c}" for i, c in enumerate(context_chunks)])
    prompt = f"""You are a scientific literature assistant. Answer the question using ONLY the provided context.
Cite the source number [1], [2], etc. for each claim.

Context:
{context}

Question: {query}

Answer:"""
    try:
        resp = gen_model.generate_content(prompt)
        return resp.text
    except Exception as e:
        return f"ERROR: {e}"

def hybrid_retrieve_top5(query):
    q_emb = model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(q_emb, embeddings)[0]
    dense_top = list(np.argsort(sims)[::-1][:50])
    tokens = query.lower().split()
    bm25_scores = bm25.get_scores(tokens)
    bm25_top = list(np.argsort(bm25_scores)[::-1][:50])
    rrf_scores = {}
    for lst in [dense_top, bm25_top]:
        for rank, idx in enumerate(lst):
            cid = chunk_ids[idx]
            rrf_scores[cid] = rrf_scores.get(cid, 0) + 1.0/(60 + rank + 1)
    top5 = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:5]
    return [id_to_text[c] for c in top5 if c in id_to_text]

print("Generating answers...")
eval_records = []
for i, query in enumerate(QUERIES):
    contexts = hybrid_retrieve_top5(query)
    answer = generate_answer(query, contexts)
    eval_records.append({
        "question": query,
        "answer": answer,
        "contexts": contexts,
        "ground_truth": query  # RAGAS uses this for answer_relevancy
    })
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(QUERIES)}")

pd.DataFrame(eval_records).to_parquet(f"{RESULTS_DIR}/eval_runs.parquet", index=False)
print(f"Saved {len(eval_records)} answers.")

In [ ]:
# ── RAGAS evaluation (FIX 2: correct Gemini wrapper) ─────────────────────────
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model="gemini-1.5-flash",
        google_api_key=GEMINI_KEY,
        convert_system_message_to_human=True
    )
)
ragas_emb = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GEMINI_KEY)
)

df_eval = pd.read_parquet(f"{RESULTS_DIR}/eval_runs.parquet")
ragas_ds = Dataset.from_dict({
    "question":    df_eval["question"].tolist(),
    "answer":      df_eval["answer"].tolist(),
    "contexts":    df_eval["contexts"].tolist(),
    "ground_truth":df_eval["ground_truth"].tolist(),
})

result = evaluate(
    ragas_ds,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_emb
)
print("\n=== RAGAS RESULTS ===")
print(result)
result.to_pandas().to_csv(f"{RESULTS_DIR}/ragas_scores.csv", index=False)
print(f"Saved: {RESULTS_DIR}/ragas_scores.csv")